# 🔬 NuPheno: Pan-Cancer Nuclear Phenotyping & Detection — Starter Guide

Welcome to the **NuPheno Benchmark Challenge**!
This challenge tasks participants with detecting cell nuclei and classifying their phenotypes across 19 human tissue types from hematoxylin and eosin (H&E) biopsy patches.

### Official Dataset Source:
- **PanNuke Dataset:** [Cancer Instance Segmentation and Classification 2 on Kaggle](https://www.kaggle.com/datasets/andrewmvd/cancer-instance-segmentation-and-classification-2)
- **Original Release:** Tissue Image Analytics (TIA) Centre, University of Warwick
- **Evaluation Metric:** $\text{mAP}@[0.50:0.95]$ (COCO Mean Average Precision)

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup project paths
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
public_data_dir = project_root / 'data' / 'public'
starter_dir = project_root / 'starter_code'
competition_dir = project_root / 'competition'

print(f'Project root: {project_root}')
print(f'Public data directory: {public_data_dir}')

## 1. Inspect Public Benchmark Manifests
The repository distributes public manifests for local development and benchmarking:
- `train_annotations.csv`: Ground-truth bounding boxes $[x, y, w, h]$ and phenotypic classes (1: Neoplastic, 2: Inflammatory, 3: Connective, 4: Dead, 5: Epithelial).
- `test_images.csv`: Test patch identifiers and organ tissue types.
- `sample_submission.csv`: Schema-compliant candidate prediction format.

In [ ]:
train_ann = pd.read_csv(public_data_dir / 'train_annotations.csv')
test_imgs = pd.read_csv(public_data_dir / 'test_images.csv')
sample_sub = pd.read_csv(public_data_dir / 'sample_submission.csv')

print(f'Training Annotations: {len(train_ann):,} nuclei across {train_ann["image_id"].nunique()} patches')
print(f'Test Images: {len(test_imgs):,} patches to evaluate')
print(f'Sample Submission rows: {len(sample_sub):,}')
train_ann.head()

## 2. Exploratory Data Analysis & Nuclei Visualizer
Run the official visualization dashboard from `starter_code/visualize_data.py` to examine class balances, cellular densities, and ground-truth spatial layouts.

In [ ]:
sys.path.append(str(starter_dir))
from visualize_data import create_pannuke_eda_dashboard

search_dirs = [project_root / 'data' / 'pannuke_raw', project_root / 'data' / 'processed_coco']
eda_output = starter_dir / 'eda_pannuke_overview.png'

create_pannuke_eda_dashboard(
    annotations_csv=public_data_dir / 'train_annotations.csv',
    search_dirs=search_dirs,
    output_path=eda_output,
    show_plot=True
)

## 3. Run the Baseline Detector
We provide a starter baseline in `starter_code/baseline.py`. It reads `test_images.csv` and generates schema-compliant candidate nuclei detections.

In [ ]:
import subprocess

baseline_script = starter_dir / 'baseline.py'
result = subprocess.run([sys.executable, str(baseline_script)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 4. Validate Submission Format
Before submitting, ensure your submission passes all verification checks (required columns, confidence ranges $[0.0, 1.0]$, bounding box coordinates within $[0, 256]$).

In [ ]:
validate_script = competition_dir / 'validate_submission.py'
sub_file = project_root / 'baseline_submission.csv'
test_file = public_data_dir / 'test_images.csv'
lb_file = project_root / 'leaderboard' / 'leaderboard.csv'

if sub_file.exists():
    res = subprocess.run([sys.executable, str(validate_script), str(sub_file), str(test_file), str(lb_file)], capture_output=True, text=True)
    print(res.stdout)
else:
    print(f'{sub_file} not found. Run baseline.py first.')

## 5. Encrypt Submission for Pull Request
All submissions must be encrypted using the official challenge 2048-bit RSA public key (`starter_code/public_key.pem`).
This ensures your predictions remain 100% confidential until auto-graded by GitHub Actions.

In [ ]:
encrypt_script = starter_dir / 'encrypt.py'
pub_key = starter_dir / 'public_key.pem'
enc_output = project_root / 'submissions' / 'baseline_submission.csv.enc'

if sub_file.exists():
    res = subprocess.run([sys.executable, str(encrypt_script), str(sub_file), str(pub_key), str(enc_output)], capture_output=True, text=True)
    print(res.stdout)
    print(f'✅ Ready to submit! Commit and push {enc_output} in a Pull Request.')
else:
    print('Generate a submission CSV before encrypting.')

## 🚀 Training on Kaggle with GPU
For full model training (Faster R-CNN, YOLOv8, RT-DETR) on the complete PanNuke dataset:
1. Open `notebooks/kaggle_pannuke_training.ipynb`.
2. Upload it to [Kaggle](https://www.kaggle.com/code).
3. In the right-hand sidebar under **Input**, click **`+ Add Input`** and search for:
   👉 `andrewmvd/cancer-instance-segmentation-and-classification-2`
4. Enable GPU accelerator (GPU T4 x2) and train your state-of-the-art detector!